# Nub Labs Playbook — Which Customers Are Similar?

**Technique:** Customer segmentation using RFM analysis and K-Means clustering

This notebook has two parts:

**Part A — Curated dataset (200 customers)**
Reproduces the exact results shown on the interactive playbook page. Run this to verify every number on the site.

**Part B — UCI Online Retail Dataset (541,909 rows)**
Runs the same RFM + K-Means analysis on real UK retailer transaction data.
Verifies that the same insight pattern holds: distinct customer segments emerge,
and the highest-AOV customers are not always the highest-frequency ones.

In [ ]:
!pip install scikit-learn pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
print('Ready.')

## Part A — Curated Dataset

200 customers with known RFM profiles. Designed to reproduce the playbook results exactly.
Each customer is defined by (recency in days, frequency in orders, monetary in dollars).

In [ ]:
def repeat_customers(recency, frequency, monetary, count):
    return [(recency, frequency, monetary)] * count

customers_raw = (
    # Champions (30)
    repeat_customers(5,  35, 8500, 4) +
    repeat_customers(8,  30, 7200, 5) +
    repeat_customers(12, 28, 6500, 5) +
    repeat_customers(7,  33, 7800, 4) +
    repeat_customers(15, 28, 6000, 5) +
    repeat_customers(10, 32, 6900, 4) +
    repeat_customers(6,  38, 9000, 3) +

    # Loyal Buyers (50)
    repeat_customers(15, 18, 2400, 7) +
    repeat_customers(25, 15, 2000, 9) +
    repeat_customers(20, 17, 2200, 8) +
    repeat_customers(35, 12, 1600, 8) +
    repeat_customers(30, 14, 1800, 9) +
    repeat_customers(45, 13, 1500, 9) +

    # At-Risk (40) — recency 110-170 days, monetary $3000-$5000
    # Clear 20-day gap above Occasional (max 90 days); differentiated by high monetary
    repeat_customers(110, 10, 5000, 6) +
    repeat_customers(130,  8, 3800, 7) +
    repeat_customers(150,  6, 3200, 7) +
    repeat_customers(160,  5, 3500, 6) +
    repeat_customers(170, 12, 4800, 7) +
    repeat_customers(120,  8, 3000, 7) +

    # Occasional (80) — recency 60-90 days, frequency 1-3, monetary $120-$320
    # Capped at 90 days so K-Means cleanly separates from At-Risk (min 110 days)
    repeat_customers(60, 2, 280, 12) +
    repeat_customers(75, 1, 150, 15) +
    repeat_customers(85, 1, 200, 12) +
    repeat_customers(80, 3, 320, 12) +
    repeat_customers(65, 2, 180, 10) +
    repeat_customers(90, 1, 120, 10) +
    repeat_customers(70, 2, 300,  9)
)

df = pd.DataFrame(customers_raw, columns=['recency', 'frequency', 'monetary'])
df.index = [f'C{i:03d}' for i in range(1, len(df) + 1)]
df.index.name = 'customer_id'

print(f'Customers: {len(df)}')
print(df.describe().round(1))


### Normalise and Cluster

Min-Max scaling puts all three features on a 0–1 scale before K-Means.
Without this, monetary (range $150–$8,500) would dominate recency (range 3–200 days).

In [ ]:
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df[['recency', 'frequency', 'monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(df_scaled)
df['cluster'] = labels

print(f'Inertia: {kmeans.inertia_:.2f}')
print(f'Silhouette score: {silhouette_score(df_scaled, labels):.3f}')
print(f'\nCluster counts:\n{df["cluster"].value_counts().sort_index()}')

### Name the Segments

K-Means assigns cluster IDs (0, 1, 2, 3) without human labels.
We assign names by examining the mean RFM profile of each cluster:
- **Champions**: high frequency + high monetary + low recency
- **At-Risk**: high monetary + high recency (gone quiet)
- **Loyal Buyers**: moderate frequency + moderate monetary
- **Occasional**: low frequency + low monetary

In [ ]:
cluster_means = df.groupby('cluster')[['recency', 'frequency', 'monetary']].mean()
cluster_means['aov'] = cluster_means['monetary'] / cluster_means['frequency'].clip(lower=0.1)
print('Cluster profiles:')
print(cluster_means.round(1))

# Assign names based on RFM profile heuristics
segment_map = {}
for cluster_id, row in cluster_means.iterrows():
    if row['frequency'] > 20:
        segment_map[cluster_id] = 'Champions'
    elif row['recency'] > 60 and row['monetary'] > 1500:
        segment_map[cluster_id] = 'At-Risk'
    elif row['frequency'] > 6:
        segment_map[cluster_id] = 'Loyal Buyers'
    else:
        segment_map[cluster_id] = 'Occasional'

df['segment'] = df['cluster'].map(segment_map)
print(f'\nSegment assignments:\n{segment_map}')
print(f'\nSegment counts:\n{df["segment"].value_counts()}')

### Verify Playbook Claims

These numbers should match the values shown on the playbook page.

In [ ]:
print('=== Verifying playbook claims ===\n')

seg_stats = df.groupby('segment').agg(
    count=('monetary', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    total_revenue=('monetary', 'sum')
).round(1)
seg_stats['aov'] = (seg_stats['avg_monetary'] / seg_stats['avg_frequency'].clip(lower=0.1)).round(0)
seg_stats['revenue_share'] = (seg_stats['total_revenue'] / seg_stats['total_revenue'].sum() * 100).round(1)
seg_stats['customer_share'] = (seg_stats['count'] / len(df) * 100).round(1)

print(seg_stats.to_string())
print()

total_rev = seg_stats['total_revenue'].sum()
print(f'Total revenue: ${total_rev:,.0f}')

# Verify key claims (may vary slightly based on K-Means convergence)
champions = seg_stats.loc['Champions'] if 'Champions' in seg_stats.index else None
at_risk = seg_stats.loc['At-Risk'] if 'At-Risk' in seg_stats.index else None
loyal = seg_stats.loc['Loyal Buyers'] if 'Loyal Buyers' in seg_stats.index else None

if champions is not None:
    print(f'\n✓ Champions: {champions["customer_share"]:.0f}% of customers, {champions["revenue_share"]:.0f}% of revenue')
if at_risk is not None and loyal is not None:
    aov_ratio = at_risk['aov'] / loyal['aov']
    print(f'✓ At-Risk AOV / Loyal AOV: ${at_risk["aov"]:.0f} / ${loyal["aov"]:.0f} = {aov_ratio:.1f}x')
if at_risk is not None:
    print(f'✓ At-Risk total revenue: ${at_risk["total_revenue"]:,.0f}')

### Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Customer Segmentation — Part A (Curated Dataset)', fontsize=14, fontweight='bold')

colors = {
    'Champions': '#EAB308',
    'Loyal Buyers': '#3B82F6',
    'At-Risk': '#EF4444',
    'Occasional': '#64748B'
}

# Scatter: Frequency vs Monetary
for seg, group in df.groupby('segment'):
    axes[0].scatter(
        group['frequency'], group['monetary'],
        c=colors.get(seg, '#888'), label=seg, alpha=0.75, s=60, edgecolors='none'
    )
axes[0].set_xlabel('Frequency (orders)', fontsize=11)
axes[0].set_ylabel('Monetary (total spend $)', fontsize=11)
axes[0].set_title('Frequency vs Monetary by Segment')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Revenue share bar chart
seg_order = [s for s in ['Champions', 'Loyal Buyers', 'At-Risk', 'Occasional'] if s in seg_stats.index]
revenue_shares = [seg_stats.loc[s, 'revenue_share'] for s in seg_order]
bar_colors = [colors.get(s, '#888') for s in seg_order]
bars = axes[1].bar(seg_order, revenue_shares, color=bar_colors, alpha=0.85, width=0.6)
axes[1].set_ylabel('Revenue Share (%)', fontsize=11)
axes[1].set_title('Revenue Share by Segment')
axes[1].grid(axis='y', alpha=0.3)
for bar, v in zip(bars, revenue_shares):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### Elbow Method — Choosing K

K=4 was chosen for this dataset. The elbow method plots inertia against K to find
the point where adding another cluster stops helping significantly.

In [ ]:
inertias = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(df_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(df_scaled, km_labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4 chosen')
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method — Inertia vs K')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(list(K_range), silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4 chosen')
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs K')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Silhouette score at K=4: {silhouette_scores[K_range.index(4)]:.3f}')

---

## Part B — UCI Online Retail Dataset

The same RFM + K-Means analysis on real UK retailer transaction data.
541,909 rows, 25,900 transactions, ~4,000 products (2010–2011).

**Goal:** verify that the same insight pattern holds on real data — distinct customer
segments emerge, and the highest-AOV customers are not always the highest-frequency ones.

**Dataset source:** https://archive.ics.uci.edu/dataset/352/online+retail

---

### Option A — Manual Upload (Colab)

1. Download `Online Retail.xlsx` from the UCI link above
2. In Colab: click the folder icon → upload the file
3. Run the cell below

### Option B — Direct Download Attempt

In [ ]:
import os

DATA_FILE = 'Online Retail.xlsx'
df_retail = None

# Option A: check if manually uploaded
if os.path.exists(DATA_FILE):
    print(f'Found {DATA_FILE} — loading...')
    df_retail = pd.read_excel(DATA_FILE, dtype={'CustomerID': str})
    print(f'Loaded {len(df_retail):,} rows')
else:
    # Option B: try downloading via UCI API
    try:
        import urllib.request
        url = 'https://archive.ics.uci.edu/static/public/352/online+retail.zip'
        print('Attempting download from UCI...')
        urllib.request.urlretrieve(url, 'online_retail.zip')
        import zipfile
        with zipfile.ZipFile('online_retail.zip', 'r') as z:
            z.extractall('.')
        # Find the Excel file
        for f in os.listdir('.'):
            if f.endswith('.xlsx'):
                df_retail = pd.read_excel(f, dtype={'CustomerID': str})
                print(f'Loaded {len(df_retail):,} rows from {f}')
                break
    except Exception as e:
        print(f'Download failed: {e}')
        print('\nPlease download manually from:')
        print('https://archive.ics.uci.edu/dataset/352/online+retail')
        print('Then upload it to Colab and re-run this cell.')

if df_retail is not None:
    print(df_retail.head())

In [ ]:
if df_retail is None:
    print('Skipping Part B — dataset not loaded. See instructions above.')
else:
    # Clean: remove nulls, cancelled orders, missing CustomerID
    df_clean = df_retail.copy()
    df_clean = df_clean.dropna(subset=['CustomerID'])
    df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]  # cancellations
    df_clean = df_clean[df_clean['Quantity'] > 0]
    df_clean = df_clean[df_clean['UnitPrice'] > 0]
    df_clean['TotalValue'] = df_clean['Quantity'] * df_clean['UnitPrice']
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

    print(f'Rows after cleaning: {len(df_clean):,}')
    print(f'Unique customers: {df_clean["CustomerID"].nunique():,}')
    print(f'Date range: {df_clean["InvoiceDate"].min()} to {df_clean["InvoiceDate"].max()}')

In [ ]:
if df_retail is not None:
    # Compute RFM
    snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

    rfm = df_clean.groupby('CustomerID').agg(
        recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
        frequency=('InvoiceNo', 'nunique'),
        monetary=('TotalValue', 'sum')
    ).reset_index()

    # Remove extreme outliers (top 1% by monetary)
    q99 = rfm['monetary'].quantile(0.99)
    rfm_clean = rfm[rfm['monetary'] <= q99].copy()

    print(f'RFM customers: {len(rfm_clean):,}')
    print(rfm_clean[['recency', 'frequency', 'monetary']].describe().round(1))

In [ ]:
if df_retail is not None:
    # Normalise and cluster
    scaler_b = MinMaxScaler()
    rfm_scaled = scaler_b.fit_transform(rfm_clean[['recency', 'frequency', 'monetary']])

    kmeans_b = KMeans(n_clusters=4, random_state=42, n_init=10)
    rfm_clean['cluster'] = kmeans_b.fit_predict(rfm_scaled)

    print(f'Silhouette score: {silhouette_score(rfm_scaled, rfm_clean["cluster"]):.3f}')

    # Name clusters
    cluster_profiles = rfm_clean.groupby('cluster')[['recency', 'frequency', 'monetary']].mean()
    cluster_profiles['aov'] = cluster_profiles['monetary'] / cluster_profiles['frequency'].clip(lower=0.1)
    print('\nCluster profiles:')
    print(cluster_profiles.round(1))

    seg_map_b = {}
    for cid, row in cluster_profiles.iterrows():
        if row['frequency'] > cluster_profiles['frequency'].median() * 1.5 and row['recency'] < cluster_profiles['recency'].median():
            seg_map_b[cid] = 'Champions'
        elif row['recency'] > cluster_profiles['recency'].median() and row['monetary'] > cluster_profiles['monetary'].median():
            seg_map_b[cid] = 'At-Risk'
        elif row['frequency'] > cluster_profiles['frequency'].median():
            seg_map_b[cid] = 'Loyal Buyers'
        else:
            seg_map_b[cid] = 'Occasional'

    rfm_clean['segment'] = rfm_clean['cluster'].map(seg_map_b)
    print(f'\nSegment counts:\n{rfm_clean["segment"].value_counts()}')

In [ ]:
if df_retail is not None:
    # Part B stats
    seg_stats_b = rfm_clean.groupby('segment').agg(
        count=('monetary', 'count'),
        avg_recency=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        avg_monetary=('monetary', 'mean'),
        total_revenue=('monetary', 'sum')
    ).round(1)
    seg_stats_b['aov'] = (seg_stats_b['avg_monetary'] / seg_stats_b['avg_frequency']).round(1)
    seg_stats_b['revenue_share'] = (seg_stats_b['total_revenue'] / seg_stats_b['total_revenue'].sum() * 100).round(1)
    print('=== Part B — Segment Statistics (UCI Online Retail) ===')
    print(seg_stats_b.to_string())

    # Verify the key insight holds
    print('\n=== Key insight verification ===')
    print('The highest-AOV customers should not be the highest-frequency ones.')
    print('Champions drive a disproportionate share of revenue.')
    print(f'\nAOV by segment:')
    print(seg_stats_b['aov'].sort_values(ascending=False))

In [ ]:
if df_retail is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Customer Segmentation — Part B (UCI Online Retail)', fontsize=14, fontweight='bold')

    colors_b = {
        'Champions': '#EAB308',
        'Loyal Buyers': '#3B82F6',
        'At-Risk': '#EF4444',
        'Occasional': '#64748B'
    }

    # Sample for scatter (avoid overplotting)
    sample = rfm_clean.sample(min(2000, len(rfm_clean)), random_state=42)
    for seg, group in sample.groupby('segment'):
        axes[0].scatter(
            group['frequency'], group['monetary'],
            c=colors_b.get(seg, '#888'), label=seg, alpha=0.5, s=30, edgecolors='none'
        )
    axes[0].set_xlabel('Frequency (orders)', fontsize=11)
    axes[0].set_ylabel('Monetary ($)', fontsize=11)
    axes[0].set_title('Frequency vs Monetary by Segment')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Revenue share
    seg_order_b = [s for s in ['Champions', 'Loyal Buyers', 'At-Risk', 'Occasional'] if s in seg_stats_b.index]
    rev_shares_b = [seg_stats_b.loc[s, 'revenue_share'] for s in seg_order_b]
    bar_colors_b = [colors_b.get(s, '#888') for s in seg_order_b]
    bars_b = axes[1].bar(seg_order_b, rev_shares_b, color=bar_colors_b, alpha=0.85, width=0.6)
    axes[1].set_ylabel('Revenue Share (%)', fontsize=11)
    axes[1].set_title('Revenue Share by Segment')
    axes[1].grid(axis='y', alpha=0.3)
    for bar, v in zip(bars_b, rev_shares_b):
        axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

---

## Next Steps

**From notebook to production:**

1. **Schedule RFM computation** — run daily in your data warehouse (BigQuery / dbt)
2. **Store segment labels** — write `customer_id → segment` to a fast lookup (Redis / Postgres)
3. **Consume in downstream systems** — marketing platform, CRM, product recommendations
4. **Monitor segment drift** — Champions moving to Loyal Buyers is an early churn signal
5. **Add features** — product category spend, geographic data, channel (email vs direct)